# Von XML zur Tabelle
In diesem Tutorial sehen wir uns an, wie wir XML-Dokumente auslesen können, und diese in eine Tabelle transformieren, mit der wir dann praktisch weiterarbeiten können. Welche Elemente wir wie ins XML übernehmen wollen, ist abhängig von der Fragestellung. In diesem Beispiel werden wir die annotierten Orte auslesen und in einer Tabelle verzeichnen. Diese Tabelle dient dann auch als Grundlage für die Kartenvisualisierung.

In [1]:
# import der relevanten Bibliotheken
from lxml import etree as et
import glob
import re

Ganz spezifisch wollen wir folgende Informationen in der Tabelle verzeichnen: 

Metadaten zum Brief
- Brief ID
- Sender:in
- Sender:in ID
- Empfänger:in
- Empfänger:in ID
- Datierung

Wir nehmen hier - etwas naiv - eine:n einzelne:n Sender:in / Empfänger:in an.

Informationen zum gefundenen Ort:
- Name des Orts
- Ort ID

Ein einfacher Weg, eine Tabelle zu bauen, ist die Informationen zuerst in einer Liste zu sammeln und die Listen dann in eine Tabelle umzuwandeln.

In [12]:
rows = []


tree = et.parse('C:/Users/alina/OneDrive/Dokumente/Uni Basel/2. Semester/(De-)Coding History/Projekt/forster1.xml')
root = tree.getroot()

namespaces = {"tei": "http://www.tei-c.org/ns/1.0"}
   

places = root.findall(".//tei:placeName", namespaces=namespaces)

for place in places:
        row = []  # in dieser Liste sammeln wir alle Infos zum jeweiligen Ort

        # etwas viel Code, aber räumt uns schön die Ortsnennung auf, auch bei Zeilenumbrüchen etc.
        place_name = et.tostring(place, method="text", encoding="utf8", with_tail=False).decode("utf8")
        place_name = re.sub(r"\s*\n\s*", " ", place_name)
        row.append(place_name)

        rows.append(row)

print(rows[:3])

[['London'], ['Madrid'], ['Goͤttingen']]


Nun haben wir eine Liste mit einer Zeile pro gefundenem Ort, diese lässt sich nun sehr einfach in eine Tabelle umwandeln. Am besten ist der Umgang mit Tabellen im externen Package [*pandas*](https://pandas.pydata.org/) zu organisieren.

Das Pandas-Package enthält eine grosse Auswahl von Operationen von tabellarischen Datensätzen, auf die aber hier erstmal nicht weiter eingegangen wird.

In [4]:
# installieren von pandas
%pip install -U pandas

  Using cached numpy-2.2.4-cp313-cp313-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.5 MB 5.1 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/11.5 MB 7.0 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.5 MB 8.9 MB/s eta 0:00:01
   -------------------------- ------------- 7.6/11.5 MB 9.8 MB/s eta 0:00:01
   ----------------------------------- ---- 10.2/11.5 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 10.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.6 MB 12.5 MB/s eta 0:00:01
   --------------- ------------------------ 5.0/12.6 MB 12.5 MB/s eta 0:00:01
   ------------------------ --------------- 7.6/12.6 MB 12.4 MB/s eta 0:00:01
   -------------------------------- ------- 10.2/12.6 MB 12.4 MB/s eta 0:00:01
   --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import pandas as pd

df = pd.DataFrame(rows, columns=['Place Name'])

print(df.head)

<bound method NDFrame.head of          Place Name
0            London
1            Madrid
2        Goͤttingen
3             Upſal
4            Danzig
...             ...
1466      Tongatabu
1467         Tahiti
1468   Oſter-Eyland
1469  Oſter-Eylands
1470    Neu-Seeland

[1471 rows x 1 columns]>


In [14]:
# speichern der Tabelle als CSV
df.to_csv('places.csv', index=False)